In [1]:
import pandas as pd
from datetime import timedelta
from tqdm import tqdm

#### Hourly Expansion

In [2]:
input_file = r"\data\smoke\extracted_smoke_data_for_10_years_pm10_coords.csv"
df = pd.read_csv(input_file, parse_dates=['Start', 'End'])

In [3]:
# Initialize an empty list to store new rows
expanded_rows = []
# Get the original row count
original_row_count = len(df)
expanded_row_count = 0  # Initialize counter for expanded rows

In [4]:
# Iterate through each row in the original data with tqdm for progress tracking
for _, row in tqdm(df.iterrows(), total=original_row_count, desc="Processing Rows"):
    start_time = row['Start']
    end_time = row['End']

    # Round up Start time to the next full hour
    start_time = (start_time + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)

    # Round up End time to the next full hour
    end_time = (end_time + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)

    # Generate hourly timestamps from rounded Start to rounded End
    current_time = start_time

    while current_time <= end_time:
        new_row = row.copy()
        new_row['DateTime'] = current_time
        expanded_rows.append(new_row)
        expanded_row_count += 1  # Increment expanded row count
        current_time += timedelta(hours=1)

# Create a new DataFrame with expanded rows
expanded_df = pd.DataFrame(expanded_rows)
print("expanded dataframe created with {} rows".format(expanded_row_count))

Processing Rows: 100%|██████████| 603374/603374 [14:51<00:00, 676.71it/s]  


expanded dataframe created with 2396374 rows


In [5]:
# Save the expanded dataset
expanded_df.to_csv(r"\data\smoke_data_10_years_pm10_expanded.csv", index=False)

#### Check for overlapping smoke data

In [7]:
expanded_df = pd.read_csv(r"\data\smoke_data_10_years_pm10_expanded.csv", parse_dates=['DateTime'])

In [8]:
#drop start data and end date from the dataframe
expanded_df = expanded_df.drop(['Start', 'End', 'Shapefile'], axis=1)

In [9]:
# Define density ranking
density_rank = {"Heavy": 3, "Medium": 2, "Light": 1}

In [10]:
expanded_df["Density_Rank"] = expanded_df["Density"].map(density_rank)

In [11]:
expanded_df.head()

,Latitude,Longitude,Density,DateTime,Density_Rank
0,39.53387,-122.190834,Light,2014-01-03 23:00:00,1
1,39.53387,-122.190834,Light,2014-01-04 00:00:00,1
2,39.53387,-122.190834,Light,2014-01-04 01:00:00,1
3,39.53387,-122.190834,Light,2014-01-04 02:00:00,1
4,39.18919,-121.998870,Light,2014-01-03 23:00:00,1


In [12]:
# Sort by Density_Rank in descending order to keep the highest rank
sorted_df = expanded_df.sort_values(by=["Latitude", "Longitude", "DateTime", "Density_Rank"], ascending=[True, True, True, False])

In [13]:
# Drop duplicates keeping the first occurrence (which has the highest rank)
filtered_df = sorted_df.drop_duplicates(subset=["Latitude", "Longitude", "DateTime"], keep="first").drop(columns=["Density_Rank"])

In [14]:
filtered_df.head()

,Latitude,Longitude,Density,DateTime
52995,25.47823,-80.483739,Light,2015-03-27 12:00:00
52996,25.47823,-80.483739,Light,2015-03-27 13:00:00
52997,25.47823,-80.483739,Light,2015-03-27 14:00:00
52998,25.47823,-80.483739,Light,2015-03-27 15:00:00
52999,25.47823,-80.483739,Light,2015-03-27 16:00:00


In [ ]:
#### Assign UID to each latitude and longitude

In [15]:
coordinates_df = pd.read_csv(r'\data\Coordinates\unique_coordinates_with_years_and_id.csv')

In [16]:
merged = pd.merge(filtered_df, coordinates_df, on=['Latitude', 'Longitude'], how='left')

In [17]:
merged.head()

,Latitude,Longitude,Density,DateTime,UID,Years_Present
0,25.47823,-80.483739,Light,2015-03-27 12:00:00,547,[2023]
1,25.47823,-80.483739,Light,2015-03-27 13:00:00,547,[2023]
2,25.47823,-80.483739,Light,2015-03-27 14:00:00,547,[2023]
3,25.47823,-80.483739,Light,2015-03-27 15:00:00,547,[2023]
4,25.47823,-80.483739,Light,2015-03-27 16:00:00,547,[2023]


In [18]:
merged = merged.drop(['Years_Present'], axis=1)

In [19]:
merged.head()

,Latitude,Longitude,Density,DateTime,UID
0,25.47823,-80.483739,Light,2015-03-27 12:00:00,547
1,25.47823,-80.483739,Light,2015-03-27 13:00:00,547
2,25.47823,-80.483739,Light,2015-03-27 14:00:00,547
3,25.47823,-80.483739,Light,2015-03-27 15:00:00,547
4,25.47823,-80.483739,Light,2015-03-27 16:00:00,547


In [ ]:
merged.to_csv("\data\smoke\Smoke_PM10_10_years.csv", index=False)